In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(r'D:\Amazon Sale Report.csv\Amazon Sale Report.csv', low_memory=False)

# 1. Drop columns we don't need
df = df.drop(columns=['Unnamed: 22', 'promotion-ids', 'fulfilled-by', 'index'])

# 2. Drop rows with missing shipping info (only 33 rows, negligible)
df = df.dropna(subset=['ship-city', 'ship-state', 'ship-postal-code', 'ship-country'])

# 3. Convert Date to proper datetime
df['Date'] = pd.to_datetime(df['Date'], format='%m-%d-%y')

# 4. Add useful derived columns
df['Month'] = df['Date'].dt.month_name()
df['Day_of_Week'] = df['Date'].dt.day_name()
df['Week'] = df['Date'].dt.isocalendar().week

# 5. Create a clean revenue column (only for orders that actually have amount)
df['Revenue'] = df['Amount']

# 6. Flag whether order was successfully fulfilled or not
successful_status = ['Shipped', 'Shipped - Delivered to Buyer', 'Shipped - Picked Up', 
                      'Shipped - Out for Delivery']
df['Order_Outcome'] = np.where(df['Status'].isin(successful_status), 'Fulfilled',
                        np.where(df['Status'] == 'Cancelled', 'Cancelled', 'Other'))

# 7. Check final shape and nulls
print(df.shape)
print(df.isnull().sum())
print(df['Order_Outcome'].value_counts())

(128942, 25)
Order ID                 0
Date                     0
Status                   0
Fulfilment               0
Sales Channel            0
ship-service-level       0
Style                    0
SKU                      0
Category                 0
Size                     0
ASIN                     0
Courier Status        6869
Qty                      0
currency              7793
Amount                7793
ship-city                0
ship-state               0
ship-postal-code         0
ship-country             0
B2B                      0
Month                    0
Day_of_Week              0
Week                     0
Revenue               7793
Order_Outcome            0
dtype: int64
Order_Outcome
Fulfilled    107558
Cancelled     18325
Other          3059
Name: count, dtype: int64


In [3]:
# ===== BASIC EDA =====

# Only use rows with valid revenue for money-related analysis
sales_df = df[df['Revenue'].notna()].copy()

print("=== Overview ===")
print("Total Orders:", df['Order ID'].nunique())
print("Total Revenue (Fulfilled+Other):", round(sales_df['Revenue'].sum(), 2))
print("Average Order Value:", round(sales_df['Revenue'].mean(), 2))
print("Date Range:", df['Date'].min(), "to", df['Date'].max())

print("\n=== Monthly Revenue Trend ===")
monthly_trend = sales_df.groupby('Month')['Revenue'].sum().sort_values(ascending=False)
print(monthly_trend)

print("\n=== Revenue by Category ===")
category_revenue = sales_df.groupby('Category')['Revenue'].agg(['sum','count']).sort_values('sum', ascending=False)
print(category_revenue)

print("\n=== Top 5 States by Revenue ===")
state_revenue = sales_df.groupby('ship-state')['Revenue'].sum().sort_values(ascending=False).head(5)
print(state_revenue)


# ===== ABC ANALYSIS =====
# Goal: classify SKUs into A (top 70-80% revenue), B (next 15%), C (remaining)

sku_revenue = sales_df.groupby('SKU')['Revenue'].sum().sort_values(ascending=False).reset_index()
sku_revenue['Cumulative_Revenue'] = sku_revenue['Revenue'].cumsum()
sku_revenue['Cumulative_Pct'] = 100 * sku_revenue['Cumulative_Revenue'] / sku_revenue['Revenue'].sum()

def classify_abc(pct):
    if pct <= 70:
        return 'A'
    elif pct <= 90:
        return 'B'
    else:
        return 'C'

sku_revenue['ABC_Class'] = sku_revenue['Cumulative_Pct'].apply(classify_abc)

print("\n=== ABC Classification Summary ===")
print(sku_revenue['ABC_Class'].value_counts())
print("\n=== Revenue Contribution by Class ===")
print(sku_revenue.groupby('ABC_Class')['Revenue'].sum())

sku_revenue.to_csv('sku_abc_classification.csv', index=False)
print("\nSaved: sku_abc_classification.csv")

=== Overview ===
Total Orders: 120350
Total Revenue (Fulfilled+Other): 78574007.3
Average Order Value: 648.57
Date Range: 2022-03-31 00:00:00 to 2022-06-29 00:00:00

=== Monthly Revenue Trend ===
Month
April    28831249.32
May      26219850.75
June     23421223.38
March      101683.85
Name: Revenue, dtype: float64

=== Revenue by Category ===
                       sum  count
Category                         
Set            39195176.03  47031
kurta          21291538.70  46700
Western Dress  11215337.69  14703
Top             5346812.30  10163
Ethnic Dress     791217.66   1093
Blouse           458408.18    881
Bottom           150667.98    420
Saree            123933.76    155
Dupatta             915.00      3

=== Top 5 States by Revenue ===
ship-state
MAHARASHTRA      13335534.14
KARNATAKA        10481114.37
TELANGANA         6916615.65
UTTAR PRADESH     6816642.08
TAMIL NADU        6515650.11
Name: Revenue, dtype: float64

=== ABC Classification Summary ===
ABC_Class
C    4063
B    1

In [4]:
# ===== ORDER FULFILLMENT & CANCELLATION ANALYSIS =====

print("=== Overall Fulfillment Rate ===")
outcome_pct = df['Order_Outcome'].value_counts(normalize=True) * 100
print(outcome_pct.round(2))

# Category-wise cancellation rate
print("\n=== Cancellation Rate by Category ===")
cat_cancel = df.groupby('Category')['Order_Outcome'].apply(
    lambda x: (x == 'Cancelled').sum() / len(x) * 100
).sort_values(ascending=False).round(2)
print(cat_cancel)

# Month-wise cancellation trend
print("\n=== Monthly Cancellation Count ===")
monthly_cancel = df.groupby(['Month', 'Order_Outcome']).size().unstack(fill_value=0)
print(monthly_cancel)

# State-wise cancellation rate (top 10 states by order volume)
print("\n=== Cancellation Rate by State (Top 10 States) ===")
top_states = df['ship-state'].value_counts().head(10).index
state_cancel = df[df['ship-state'].isin(top_states)].groupby('ship-state')['Order_Outcome'].apply(
    lambda x: (x == 'Cancelled').sum() / len(x) * 100
).sort_values(ascending=False).round(2)
print(state_cancel)

# Fulfillment type analysis (Amazon Fulfilled vs Merchant)
print("\n=== Fulfillment Type vs Outcome ===")
fulfil_outcome = pd.crosstab(df['Fulfilment'], df['Order_Outcome'], normalize='index') * 100
print(fulfil_outcome.round(2))

# B2B vs B2C
print("\n=== B2B vs B2C Order Split ===")
print(df['B2B'].value_counts())
b2b_revenue = sales_df.groupby('B2B')['Revenue'].sum()
print("\nRevenue split:")
print(b2b_revenue)

# Save cancellation data
cancel_df = df[df['Order_Outcome'] == 'Cancelled'].groupby(['Category','ship-state']).size().reset_index(name='Cancelled_Orders')
cancel_df.to_csv('cancellation_analysis.csv', index=False)
print("\nSaved: cancellation_analysis.csv")

=== Overall Fulfillment Rate ===
Order_Outcome
Fulfilled    83.42
Cancelled    14.21
Other         2.37
Name: proportion, dtype: float64

=== Cancellation Rate by Category ===
Category
Set              14.59
kurta            14.54
Western Dress    13.69
Bottom           13.64
Saree            12.80
Blouse           12.53
Ethnic Dress     12.51
Top              12.02
Dupatta           0.00
Name: Order_Outcome, dtype: float64

=== Monthly Cancellation Count ===
Order_Outcome  Cancelled  Fulfilled  Other
Month                                     
April               7133      41018    903
June                5301      30963   1425
March                 18        152      1
May                 5873      35425    730

=== Cancellation Rate by State (Top 10 States) ===
ship-state
KERALA            17.84
ANDHRA PRADESH    16.43
UTTAR PRADESH     15.08
WEST BENGAL       14.82
TELANGANA         14.42
TAMIL NADU        13.88
MAHARASHTRA       13.32
Gujarat           13.17
DELHI             13.09

In [5]:
import pandas as pd
import numpy as np

# ===== STATE-WISE SALES ANALYSIS =====

print("=== Top 15 States by Revenue ===")
state_analysis = sales_df.groupby('ship-state').agg(
    Total_Revenue=('Revenue', 'sum'),
    Total_Orders=('Order ID', 'count'),
    Avg_Order_Value=('Revenue', 'mean')
).sort_values('Total_Revenue', ascending=False).head(15).round(2)
print(state_analysis)

print("\n=== Top 10 Cities by Revenue ===")
city_revenue = sales_df.groupby('ship-city')['Revenue'].sum().sort_values(ascending=False).head(10).round(2)
print(city_revenue)

print("\n=== Size Distribution by Category ===")
size_dist = df.groupby(['Category', 'Size']).size().reset_index(name='Order_Count')
size_top = size_dist.sort_values('Order_Count', ascending=False).head(20)
print(size_top)


# ===== POWER BI EXPORT - MASTER CLEANED FILE =====
print("\n--- Exporting Power BI ready files ---")

# 1. Main transactions file (for all visuals)
powerbi_main = df[[
    'Order ID', 'Date', 'Month', 'Day_of_Week', 'Week',
    'Status', 'Order_Outcome', 'Fulfilment',
    'Category', 'Size', 'SKU', 'Style',
    'Qty', 'Amount', 'Revenue',
    'ship-city', 'ship-state', 'B2B'
]].copy()
powerbi_main.columns = [
    'Order_ID', 'Date', 'Month', 'Day_of_Week', 'Week',
    'Status', 'Order_Outcome', 'Fulfilment_Type',
    'Category', 'Size', 'SKU', 'Style',
    'Quantity', 'Amount', 'Revenue',
    'City', 'State', 'Is_B2B'
]
powerbi_main.to_csv('powerbi_main_data.csv', index=False)
print("Saved: powerbi_main_data.csv - rows:", len(powerbi_main))

# 2. ABC Classification file (already saved earlier)
# sku_abc_classification.csv - already done

# 3. Monthly summary
monthly_summary = sales_df.groupby('Month').agg(
    Total_Revenue=('Revenue', 'sum'),
    Total_Orders=('Order ID', 'count'),
    Avg_Order_Value=('Revenue', 'mean')
).round(2).reset_index()
monthly_summary.to_csv('powerbi_monthly_summary.csv', index=False)
print("Saved: powerbi_monthly_summary.csv")

# 4. Category summary
category_summary = sales_df.groupby('Category').agg(
    Total_Revenue=('Revenue', 'sum'),
    Total_Orders=('Order ID', 'count'),
    Avg_Order_Value=('Revenue', 'mean')
).round(2).reset_index()
category_summary.to_csv('powerbi_category_summary.csv', index=False)
print("Saved: powerbi_category_summary.csv")

# 5. State summary (with cancellation rate)
state_orders = df.groupby('ship-state').size().reset_index(name='Total_Orders')
state_cancelled = df[df['Order_Outcome']=='Cancelled'].groupby('ship-state').size().reset_index(name='Cancelled_Orders')
state_revenue_df = sales_df.groupby('ship-state')['Revenue'].sum().reset_index(name='Total_Revenue')

state_summary = state_orders.merge(state_cancelled, on='ship-state', how='left')
state_summary = state_summary.merge(state_revenue_df, on='ship-state', how='left')
state_summary['Cancellation_Rate_Pct'] = (state_summary['Cancelled_Orders'] / state_summary['Total_Orders'] * 100).round(2)
state_summary.to_csv('powerbi_state_summary.csv', index=False)
print("Saved: powerbi_state_summary.csv")

print("\n✅ ALL FILES EXPORTED SUCCESSFULLY!")
print("Files ready for Power BI:")
print("1. powerbi_main_data.csv       → Main transactions (all visuals)")
print("2. sku_abc_classification.csv  → ABC Analysis")
print("3. powerbi_monthly_summary.csv → Monthly trend")
print("4. powerbi_category_summary.csv → Category performance")
print("5. powerbi_state_summary.csv   → State-wise sales + cancellation")

=== Top 15 States by Revenue ===
                Total_Revenue  Total_Orders  Avg_Order_Value
ship-state                                                  
MAHARASHTRA       13335534.14         21073           632.83
KARNATAKA         10481114.37         16394           639.33
TELANGANA          6916615.65         10637           650.24
UTTAR PRADESH      6816642.08          9947           685.30
TAMIL NADU         6515650.11         10809           602.80
DELHI              4235215.97          6393           662.48
KERALA             3830227.58          6151           622.70
WEST BENGAL        3507880.44          5547           632.39
ANDHRA PRADESH     3219831.72          5055           636.96
HARYANA            2882092.99          4188           688.18
Gujarat            2728651.82          4153           657.03
RAJASTHAN          1716802.40          2468           695.62
MADHYA PRADESH     1592382.98          2367           672.74
BIHAR              1394388.32          1935         